# LangChain Level 1a：工程进度知识库的可靠检索

这份 notebook 只针对当前 **8 份输变电工程进度 TXT**。它在 level1 的朴素召回上，
增加这个小场景真正需要的约束：

**查询规范化 → 工程 metadata route → 中文 BM25 → Dense 补充 → Weighted RRF
→ 完整任务记录定位 → Evidence Gate → 引用回答或拒答**

所有项目示例代码都直接写在本 notebook 中，不导入 configs.py、
project_progress_reliable.py 或 text_retrieval.py。为保持可读性，查询意图、任务记录、
trace 和结果都使用普通 dict，不建立知识库 class 或多层 adapter。

## 1. 导入、路径与真实语料

只支持从仓库根目录或 ZZworkbench 启动。InMemoryVectorStore 足够容纳当前几十个
chunk；本实验不建立持久化向量数据库，也不调用 LLM。

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi


repo_root = Path.cwd().resolve()
if repo_root.name == "ZZworkbench":
    repo_root = repo_root.parent

text_dir = repo_root / "knowledge" / "project_progress" / "texts" / "v4"
eval_dir = repo_root / "knowledge" / "project_progress" / "evals"
embed_model_name = "iic--nlp_gte_sentence-embedding_chinese-base"
embed_path = Path("/mnt/e/local_models/embedding") / embed_model_name

assert text_dir.is_dir(), f"找不到语料目录：{text_dir}"
assert eval_dir.is_dir(), f"找不到评测目录：{eval_dir}"
assert embed_path.is_dir(), "找不到本地 embedding 模型目录"
print({"repo_ok": repo_root.name == "pipelines_rag", "model": embed_path.name})

{'repo_ok': True, 'model': 'iic--nlp_gte_sentence-embedding_chinese-base'}


## 2. 加载并切分文档

metadata 只保留后续实际使用的来源、标题、版本、稳定 ID 和字符位置。
chunk_size=872、overlap=160 是当前语料的实验设置；是否可靠要由下面的
“541 条任务记录能否回指 chunk”审计判断。

In [2]:
title_pattern = re.compile(r"该进度计划的完整名称为(?P<title>.+?)[。\n]")
documents: list[Document] = []

for path in sorted(text_dir.glob("*.txt")):
    text = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n").strip()
    relative_source = path.relative_to(repo_root).as_posix()
    document_id = "doc-" + hashlib.sha1(
        f"{relative_source}\n{text}".encode("utf-8")
    ).hexdigest()[:16]
    title_match = title_pattern.search(text)
    title = title_match.group("title").strip() if title_match else path.stem
    documents.append(
        Document(
            id=document_id,
            page_content=text,
            metadata={
                "source": relative_source,
                "source_name": path.name,
                "version": path.parent.name,
                "document_id": document_id,
                "title": title,
            },
        )
    )

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""],
    keep_separator="end",
    chunk_size=872,
    chunk_overlap=160,
    add_start_index=True,
    strip_whitespace=True,
)

chunks: list[Document] = []
for document in documents:
    for chunk_index, chunk in enumerate(splitter.split_documents([document])):
        start_index = int(chunk.metadata.get("start_index", 0))
        chunk_id = (
            f"{document.metadata['document_id']}:"
            f"{start_index:06d}:{chunk_index:03d}"
        )
        chunks.append(
            Document(
                id=chunk_id,
                page_content=chunk.page_content,
                metadata={
                    **chunk.metadata,
                    "chunk_id": chunk_id,
                    "chunk_index": chunk_index,
                    "start_index": start_index,
                },
            )
        )

print({"documents": len(documents), "chunks": len(chunks)})
assert len(documents) == 8

{'documents': 8, 'chunks': 63}


## 3. 解析完整任务记录

Retriever 的返回单位是 chunk，但工程事实的判断单位是完整任务记录。当前 TXT 已整理成
稳定句式，因此用几条可审计的正则提取 task_id、task_name、父子关系、日期、工期和页码。

每条记录还要回指包含它的 chunk。若出现记录无法定位到 chunk，应该先修切分/数据，
而不是继续调相似度参数。

In [3]:
page_pattern = re.compile(r"以下内容来自PDF第(?P<page>\d+)页。")
task_id_pattern = re.compile(r"^标识号(?P<task_id>\d+)是")
parent_id_pattern = re.compile(r"[（(]标识号(?P<parent_id>\d+)[）)]")
quoted_pattern = re.compile(r"“(?P<value>[^”]+)”")
duration_pattern = re.compile(r"工期(?P<value>[^。]+)")
start_pattern = re.compile(r"计划开始(?P<value>\d{4}年\d{1,2}月\d{1,2}日)")
end_pattern = re.compile(r"计划完成(?P<value>\d{4}年\d{1,2}月\d{1,2}日)")

chunks_by_document: dict[str, list[Document]] = defaultdict(list)
for chunk in chunks:
    chunks_by_document[str(chunk.metadata["document_id"])].append(chunk)

records: list[dict] = []
for document in documents:
    page: int | None = None
    document_id = str(document.metadata["document_id"])
    document_chunks = chunks_by_document[document_id]

    for paragraph in re.split(r"\n\s*\n", document.page_content):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if page_match := page_pattern.fullmatch(paragraph):
            page = int(page_match.group("page"))
            continue
        task_match = task_id_pattern.match(paragraph)
        if task_match is None:
            continue

        task_id = task_match.group("task_id")
        header = paragraph.split("。", 1)[0]
        quoted = [match.group("value") for match in quoted_pattern.finditer(header)]
        if not quoted:
            continue

        task_name = quoted[-1]
        parent_task = quoted[-2] if len(quoted) > 1 else None
        parent_match = parent_id_pattern.search(header)
        is_child = "子任务" in header
        is_parent = "同时是父任务" in header or (
            header.startswith(f"标识号{task_id}是父任务") and not is_child
        )
        relation = (
            "top_level"
            if "顶层独立任务" in header
            else "child_parent"
            if is_child and is_parent
            else "child"
            if is_child
            else "parent"
            if is_parent
            else "task"
        )

        containing_chunk = next(
            (chunk for chunk in document_chunks if paragraph in chunk.page_content),
            None,
        )
        if containing_chunk is None:
            marker = f"标识号{task_id}是"
            containing_chunk = next(
                (
                    chunk
                    for chunk in document_chunks
                    if marker in chunk.page_content and task_name in chunk.page_content
                ),
                None,
            )

        duration_match = duration_pattern.search(paragraph)
        start_match = start_pattern.search(paragraph)
        end_match = end_pattern.search(paragraph)
        records.append(
            {
                "record_id": f"{document_id}:{task_id}",
                "source": document.metadata["source"],
                "source_name": document.metadata["source_name"],
                "document_id": document_id,
                "chunk_id": str(containing_chunk.id) if containing_chunk else None,
                "project_title": document.metadata["title"],
                "page": page,
                "task_id": task_id,
                "task_name": task_name,
                "parent_task": parent_task,
                "parent_task_id": (
                    parent_match.group("parent_id") if parent_match else None
                ),
                "relation": relation,
                "is_parent": is_parent,
                "is_child": is_child,
                "duration": (
                    duration_match.group("value").strip()
                    if duration_match
                    else None
                ),
                "start_date": start_match.group("value") if start_match else None,
                "end_date": end_match.group("value") if end_match else None,
                "raw_text": paragraph,
            }
        )

records_without_chunk = [record for record in records if not record["chunk_id"]]
print(
    {
        "records": len(records),
        "records_without_chunk": len(records_without_chunk),
    }
)
assert len(records) == 541
assert not records_without_chunk

{'records': 541, 'records_without_chunk': 0}


## 4. 查询规范化与工程 route

当前场景中的工程名是强约束。别名表只覆盖真实 8 份文档及评测问题中的表达，
不让 embedding 猜工程。若多个文档得到相同的最长别名匹配，则都保留，后续由任务记录消歧。

In [4]:
source_aliases = {
    "110kV黄金输变电工程三级进度计划.txt": [
        "黄金",
        "黄金输变电工程",
        "珠海黄金输变电工程",
        "110千伏黄金输变电工程",
        "110kV黄金输变电工程",
    ],
    "110千伏节点计划-重点关注.txt": [
        "110千伏节点计划",
        "110千伏输变电工程节点计划",
        "输变电工程节点计划",
        "节点计划",
    ],
    "三级进度计划-土建.txt": [
        "禾益",
        "禾益输变电工程",
        "禾益输变电工程变电站土建",
        "珠海110千伏禾益输变电工程",
    ],
    "三虎输变电工程三级进度计划土建部分.txt": [
        "三虎",
        "三虎输变电工程",
        "三虎土建",
        "三虎输变电工程土建部分",
        "珠海110千伏三虎输变电工程土建部分",
    ],
    "三虎输变电工程三级进度计划电气部分.txt": [
        "三虎",
        "三虎输变电工程",
        "三虎电气",
        "三虎输变电工程电气部分",
        "珠海110千伏三虎输变电工程电气部分",
    ],
    "南溪三级进度.txt": [
        "南溪",
        "南溪输变电工程",
        "南溪旅游输变电工程",
        "南溪（旅游）输变电工程",
        "珠海110千伏南溪（旅游）输变电工程",
    ],
    "珠海110kV江湾输变电工程总体进度计划横道图.txt": [
        "江湾",
        "江湾输变电工程",
        "江湾总体计划",
        "江湾输变电工程总体计划",
        "江湾110千伏输变电工程总体计划",
        "珠海江湾110千伏输变电工程总体计划",
    ],
    "珠海110千伏江湾输变电工程施工进度计划（202.txt": [
        "江湾",
        "江湾输变电工程",
        "江湾施工进度计划",
        "江湾输变电工程施工进度计划",
        "珠海110千伏江湾输变电工程施工进度计划",
    ],
}

titles = {
    document.metadata["source_name"]: document.metadata["title"]
    for document in documents
}
for source_name, title in titles.items():
    source_aliases[source_name] = list(
        dict.fromkeys(
            [title, source_name.rsplit(".", 1)[0], *source_aliases[source_name]]
        )
    )


def normalize_query(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", text).casefold()
    normalized = re.sub(
        r"(?P<voltage>\d+)\s*k\s*v",
        r"\g<voltage>千伏",
        normalized,
    )
    normalized = normalized.replace("签订", "签定")
    return re.sub(r"[^0-9a-z\u3400-\u9fff]+", "", normalized)


def resolve_project_sources(query: str) -> list[str]:
    normalized_query = normalize_query(query)
    scores = {}
    for source_name, aliases in source_aliases.items():
        lengths = [
            len(normalized_alias)
            for alias in aliases
            if (normalized_alias := normalize_query(alias))
            and normalized_alias in normalized_query
        ]
        scores[source_name] = max(lengths, default=0)
    best = max(scores.values(), default=0)
    return sorted(
        source_name
        for source_name, score in scores.items()
        if best > 0 and score == best
    )

## 5. QueryIntent 使用普通 dict

先在 route 后的记录中查找问题里明确出现的最长任务名；只有精确名称失败时才做保守的
问句后缀剥离。requested_fields 决定 Evidence Gate 必须在同一条记录中看到哪些字段。

In [5]:
def parse_intent(query: str) -> dict:
    if not query.strip():
        raise ValueError("query cannot be empty")

    normalized = normalize_query(query)
    project_sources = resolve_project_sources(query)
    scoped_records = [
        record
        for record in records
        if not project_sources or record["source_name"] in project_sources
    ]
    exact_task_names = {
        record["task_name"]
        for record in scoped_records
        if normalize_query(record["task_name"]) in normalized
    }
    task_hint = (
        max(exact_task_names, key=lambda name: len(normalize_query(name)))
        if exact_task_names
        else None
    )

    if task_hint is None and len(project_sources) == 1 and "总体计划" in normalized:
        root_records = [
            record
            for record in scoped_records
            if record["task_id"] == "1" and record["parent_task_id"] is None
        ]
        if len(root_records) == 1:
            task_hint = root_records[0]["task_name"]

    if task_hint is None:
        segment = unicodedata.normalize("NFKC", query).strip().split("的")[-1]
        suffixes = [
            r"计划开始、完成和工期分别是什么.*$",
            r"计划的开始和完成日期是什么.*$",
            r"计划起止时间是什么.*$",
            r"计划什么时候完成.*$",
            r"计划什么时候开始.*$",
            r"什么时候完成.*$",
            r"什么时候开始.*$",
            r"计划多久.*$",
            r"在什么时间.*$",
        ]
        for suffix in suffixes:
            segment = re.sub(suffix, "", segment).strip("？?。 ，,")
        task_hint = re.sub(r"计划$", "", segment).strip() or None

    asks_duration = "多久" in normalized or "工期" in normalized
    asks_start = any(
        phrase in normalized
        for phrase in ["开始", "起止", "在什么时间", "什么时间段", "多久"]
    )
    asks_end = any(
        phrase in normalized
        for phrase in ["完成", "结束", "起止", "在什么时间", "什么时间段", "多久"]
    )
    requested_fields = []
    if asks_start:
        requested_fields.append("start_date")
    if asks_end:
        requested_fields.append("end_date")
    if asks_duration:
        requested_fields.append("duration")
    if not requested_fields:
        requested_fields = ["start_date", "end_date"]

    return {
        "original_query": query,
        "normalized_query": normalized,
        "project_sources": project_sources,
        "task_hint": task_hint,
        "requested_fields": requested_fields,
    }


for example in [
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "施工准备什么时候完成？",
]:
    print(json.dumps(parse_intent(example), ensure_ascii=False))

{"original_query": "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？", "normalized_query": "珠海110千伏黄金输变电工程的主体结构封顶计划什么时候完成", "project_sources": ["110kV黄金输变电工程三级进度计划.txt"], "task_hint": "主体结构封顶", "requested_fields": ["end_date"]}
{"original_query": "施工准备什么时候完成？", "normalized_query": "施工准备什么时候完成", "project_sources": [], "task_hint": "施工准备", "requested_fields": ["end_date"]}


## 6. 中文 BM25 与 Dense

BM25 负责工程名、任务名、编号和日期等词面锚点；Dense 补充口语改写。
两个分支都先应用同一组 project_sources。

In [6]:
lexical_pattern = re.compile(
    r"[\u3400-\u9fff]+|[a-z0-9]+(?:[._/-][a-z0-9]+)*"
)


def tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).casefold()
    tokens: list[str] = []
    for match in lexical_pattern.finditer(normalized):
        value = match.group(0)
        if "\u3400" <= value[0] <= "\u9fff":
            if len(value) == 1:
                tokens.append(value)
            else:
                tokens.extend(value[i : i + 2] for i in range(len(value) - 1))
                tokens.extend(value[i : i + 3] for i in range(len(value) - 2))
        else:
            tokens.append(value)
    return tokens


tokenized_chunks = [
    tokenize(f"{chunk.metadata['title']}\n{chunk.page_content}")
    for chunk in chunks
]
bm25 = BM25Okapi(tokenized_chunks)

embed_model = HuggingFaceEmbeddings(
    model=str(embed_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    query_encode_kwargs={"normalize_embeddings": True},
    show_progress=False,
)
vector_store = InMemoryVectorStore(embedding=embed_model)
vector_store.add_documents(
    documents=chunks,
    ids=[str(chunk.id) for chunk in chunks],
)
print({"bm25_chunks": len(tokenized_chunks), "dense_chunks": len(chunks)})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'bm25_chunks': 63, 'dense_chunks': 63}


In [7]:
def bm25_search(query: str, allowed_sources: set[str] | None, k: int = 20) -> list[dict]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(
        (
            (index, float(score))
            for index, score in enumerate(scores)
            if (
                allowed_sources is None
                or chunks[index].metadata["source_name"] in allowed_sources
            )
            and float(score) > 0
        ),
        key=lambda item: (-item[1], str(chunks[item[0]].id)),
    )[:k]
    return [
        {"document": chunks[index], "score": score, "rank": rank}
        for rank, (index, score) in enumerate(ranked, start=1)
    ]


def dense_search(query: str, allowed_sources: set[str] | None, k: int = 20) -> list[dict]:
    source_filter = lambda doc: (
        allowed_sources is None
        or doc.metadata["source_name"] in allowed_sources
    )
    pairs = vector_store.similarity_search_with_score(
        query,
        k=k,
        filter=source_filter,
    )
    return [
        {"document": doc, "score": float(score), "rank": rank}
        for rank, (doc, score) in enumerate(pairs, start=1)
    ]

## 7. Weighted RRF

BM25 score 与 cosine similarity 不在同一尺度，融合时只使用排名。
当前数据强依赖精确工程名和任务名，因此 BM25 权重 0.65、Dense 权重 0.35；
这是本知识库的评测起点，不是通用最优参数。

In [8]:
def weighted_rrf(
    lexical_hits: list[dict],
    dense_hits: list[dict],
    k: int = 8,
    rrf_k: int = 60,
) -> list[dict]:
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, Document] = {}
    signals: dict[str, dict] = defaultdict(dict)

    for branch, weight, hits in [
        ("lexical", 0.65, lexical_hits),
        ("dense", 0.35, dense_hits),
    ]:
        for rank, hit in enumerate(hits, start=1):
            chunk_id = str(hit["document"].id)
            docs[chunk_id] = hit["document"]
            scores[chunk_id] += weight / (rrf_k + rank)
            signals[chunk_id][f"{branch}_rank"] = rank
            signals[chunk_id][f"{branch}_score"] = hit["score"]

    ordered = sorted(scores, key=lambda chunk_id: (-scores[chunk_id], chunk_id))[:k]
    return [
        {
            "document": docs[chunk_id],
            "score": scores[chunk_id],
            "rank": rank,
            **signals[chunk_id],
        }
        for rank, chunk_id in enumerate(ordered, start=1)
    ]

## 8. 一个入口：retrieve + record lookup + Evidence Gate

query_schedule 是唯一编排入口。Retriever 只产生候选；随后使用完整 records 做唯一性、
字段、引用和候选覆盖检查。只有 exact 才格式化答案。

In [9]:
field_labels = {
    "start_date": "计划开始",
    "end_date": "计划完成",
    "duration": "工期",
}


def query_schedule(query: str, final_k: int = 8) -> dict:
    intent = parse_intent(query)
    allowed_sources = (
        set(intent["project_sources"]) if intent["project_sources"] else None
    )
    lexical_hits = bm25_search(
        intent["normalized_query"],
        allowed_sources,
        k=20,
    )
    dense_hits = dense_search(
        intent["normalized_query"],
        allowed_sources,
        k=20,
    )
    fused_hits = weighted_rrf(lexical_hits, dense_hits, k=final_k)
    retrieved_chunk_ids = {
        str(hit["document"].id) for hit in fused_hits
    }

    trace = {
        "project_sources": intent["project_sources"],
        "lexical_candidates": len(lexical_hits),
        "dense_candidates": len(dense_hits),
        "fused_chunk_ids": [str(hit["document"].id) for hit in fused_hits],
    }

    if not intent["task_hint"]:
        return {
            "status": "insufficient",
            "answer": "问题中没有可确认的任务名称，请补充具体任务。",
            "intent": intent,
            "records": [],
            "trace": trace,
        }

    scoped_records = [
        record
        for record in records
        if not intent["project_sources"]
        or record["source_name"] in intent["project_sources"]
    ]
    normalized_task = normalize_query(intent["task_hint"])
    matches = [
        record
        for record in scoped_records
        if normalize_query(record["task_name"]) == normalized_task
    ]

    if "父任务" in intent["normalized_query"]:
        parent_matches = [record for record in matches if record["is_parent"]]
        if parent_matches:
            matches = parent_matches
    elif "顶层" in intent["normalized_query"]:
        top_matches = [record for record in matches if record["relation"] == "top_level"]
        if top_matches:
            matches = top_matches

    if not matches:
        return {
            "status": "not_found",
            "answer": f"知识库中没有找到任务“{intent['task_hint']}”的可验证记录。",
            "intent": intent,
            "records": [],
            "trace": trace,
        }

    if len({record["source_name"] for record in matches}) > 1:
        sources = "、".join(sorted({record["source_name"] for record in matches}))
        return {
            "status": "ambiguous",
            "answer": (
                f"任务“{intent['task_hint']}”出现在多个工程文档中，"
                f"请补充工程范围：{sources}"
            ),
            "intent": intent,
            "records": matches,
            "trace": trace,
        }

    if len(matches) > 1:
        task_ids = "、".join(record["task_id"] for record in matches)
        return {
            "status": "ambiguous",
            "answer": (
                f"同一文档中有多个“{intent['task_hint']}”任务"
                f"（标识号{task_ids}），请补充父子层级。"
            ),
            "intent": intent,
            "records": matches,
            "trace": trace,
        }

    record = matches[0]
    missing_fields = [
        field
        for field in intent["requested_fields"]
        if not record.get(field)
    ]
    if missing_fields:
        return {
            "status": "insufficient",
            "answer": (
                f"已找到任务“{record['task_name']}”，"
                f"但证据缺少字段：{', '.join(missing_fields)}。"
            ),
            "intent": intent,
            "records": [record],
            "trace": {**trace, "missing_fields": missing_fields},
        }

    missing_citations = [
        field
        for field in ["source_name", "task_id", "chunk_id"]
        if not record.get(field)
    ]
    if missing_citations:
        return {
            "status": "insufficient",
            "answer": "已定位任务记录，但引用元数据不完整，不能输出确定答案。",
            "intent": intent,
            "records": [record],
            "trace": {**trace, "missing_citations": missing_citations},
        }

    if record["chunk_id"] not in retrieved_chunk_ids:
        return {
            "status": "insufficient",
            "answer": (
                f"已定位任务“{record['task_name']}”，"
                "但本次候选召回未覆盖其证据记录。"
            ),
            "intent": intent,
            "records": [record],
            "trace": {**trace, "record_in_candidates": False},
        }

    facts = "，".join(
        f"{field_labels[field]}{record[field]}"
        for field in intent["requested_fields"]
    )
    citation = (
        f"来源：{record['source_name']}，任务标识号{record['task_id']}"
        + (
            f"，PDF第{record['page']}页"
            if record["page"] is not None
            else ""
        )
        + f"，chunk_id={record['chunk_id']}"
    )
    return {
        "status": "exact",
        "answer": (
            f"{record['project_title']}中，"
            f"“{record['task_name']}”{facts}。\n{citation}"
        ),
        "intent": intent,
        "records": [record],
        "trace": {**trace, "record_in_candidates": True},
    }

## 9. 成功、歧义、不存在与父任务

这些都是当前 v4 中可直接核验的真实路径。拒答是正常结果，不是异常。

In [10]:
example_queries = [
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "施工准备什么时候完成？",
    "南溪输变电工程的锅炉点火计划什么时候完成？",
    "110千伏节点计划中父任务试桩什么时候完成？",
]

for query in example_queries:
    result = query_schedule(query)
    print(
        json.dumps(
            {
                "query": query,
                "status": result["status"],
                "answer": result["answer"],
            },
            ensure_ascii=False,
            indent=2,
        )
    )

{
  "query": "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
  "status": "exact",
  "answer": "珠海110千伏黄金输变电工程工程进度计划横道图中，“主体结构封顶”计划完成2024年10月24日。\n来源：110kV黄金输变电工程三级进度计划.txt，任务标识号10，PDF第1页，chunk_id=doc-b94ad48bc28360c5:000000:000"
}
{
  "query": "施工准备什么时候完成？",
  "status": "ambiguous",
  "answer": "任务“施工准备”出现在多个工程文档中，请补充工程范围：110kV黄金输变电工程三级进度计划.txt、三级进度计划-土建.txt、三虎输变电工程三级进度计划土建部分.txt、珠海110千伏江湾输变电工程施工进度计划（202.txt"
}


{
  "query": "南溪输变电工程的锅炉点火计划什么时候完成？",
  "status": "not_found",
  "answer": "知识库中没有找到任务“锅炉点火”的可验证记录。"
}


{
  "query": "110千伏节点计划中父任务试桩什么时候完成？",
  "status": "exact",
  "answer": "110千伏输变电工程节点计划中，“试桩”计划完成2028年2月29日。\n来源：110千伏节点计划-重点关注.txt，任务标识号4，PDF第1页，chunk_id=doc-eb38ed0111478bb9:000000:000"
}


## 10. 反例：已定位记录，但候选没有证据

把 final_k 缩到 0，模拟 retriever 未覆盖目标记录。结构化 records 仍能找到任务，
但 Evidence Gate 必须返回 insufficient，不能绕过候选召回直接回答。

In [11]:
missed_evidence = query_schedule(
    "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    final_k=0,
)
print(
    {
        "status": missed_evidence["status"],
        "answer": missed_evidence["answer"],
        "record_in_candidates": missed_evidence["trace"].get(
            "record_in_candidates"
        ),
    }
)
assert missed_evidence["status"] == "insufficient"

{'status': 'insufficient', 'answer': '已定位任务“主体结构封顶”，但本次候选召回未覆盖其证据记录。', 'record_in_candidates': False}


## 11. 运行 13 条真实评测

retrieval_v4 有 8 条事实查询，reliability_v4 有 5 条 exact/ambiguous/not_found/
父任务/全字段查询。评测检查状态、来源、证据词和结构化字段，并打印失败明细。

In [12]:
def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


cases = [
    *load_jsonl(eval_dir / "retrieval_v4.jsonl"),
    *load_jsonl(eval_dir / "reliability_v4.jsonl"),
]
evaluation_rows = []

for case in cases:
    result = query_schedule(case["query"])
    expected_status = case.get("expected_status", "exact")
    expected_source = case.get("expected_source")
    expected_terms = [str(term) for term in case.get("expected_terms", [])]
    expected_fields = {
        str(field): str(value)
        for field, value in case.get("expected_fields", {}).items()
    }
    evidence = "\n".join(
        record["raw_text"] for record in result["records"]
    )
    status_ok = result["status"] == expected_status
    source_ok = expected_source is None or any(
        record["source_name"] == expected_source
        for record in result["records"]
    )
    terms_ok = all(term in evidence for term in expected_terms)
    fields_ok = all(
        any(str(record.get(field)) == value for record in result["records"])
        for field, value in expected_fields.items()
    )
    evaluation_rows.append(
        {
            "id": case["id"],
            "status": result["status"],
            "passed": status_ok and source_ok and terms_ok and fields_ok,
        }
    )

failures = [row for row in evaluation_rows if not row["passed"]]
print(
    {
        "cases": len(evaluation_rows),
        "passed": sum(row["passed"] for row in evaluation_rows),
        "pass_rate": round(
            sum(row["passed"] for row in evaluation_rows)
            / len(evaluation_rows),
            4,
        ),
    }
)
print("failures:", failures)
assert not failures

{'cases': 13, 'passed': 13, 'pass_rate': 1.0}
failures: []


## 12. 结论

- 工程 route、完整任务记录和 Evidence Gate 是当前数据中的必要约束，不是为了展示高级组件；
- BM25/Dense/RRF 只负责候选召回，唯一记录及字段校验才决定是否回答；
- 全部项目逻辑在 notebook 内，调用链只有明确的处理阶段；
- 13/13 只代表当前小知识库的防回归结果，不代表通用 RAG 准确率；
- 数据量仍很小，InMemoryVectorStore 是更合适的工程选择。